<a href="https://colab.research.google.com/github/ashikjoel/-Context-Aware-Neural-Recommendation-Engine/blob/ashik_joel/Context_Aware_Neural_Recommendation_Engine(week_1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#Install PySpark  (used to process large dataset)

In [ ]:
!pip install -q pyspark

#Import PySpark  (let us to work with spark)

In [ ]:
from pyspark.sql import SparkSession

#Create the Spark Session

In [ ]:
spark = (
    SparkSession.builder
    .appName("HM-Recommendation-System")
    .getOrCreate()
)

In [ ]:
print("Spark Version:", spark.version)

Spark Version: 4.0.3


#Downloading the H&M Dataset from kaggle

In [ ]:
from google.colab import files
uploaded = files.upload()


In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle competitions download -c h-and-m-personalized-fashion-recommendations

100% 28.7G/28.7G [04:23<00:00, 117MB/s]



In [ ]:
!unzip -q h-and-m-personalized-fashion-recommendations.zip -d hm_dataset

In [ ]:
import os

print(os.listdir("hm_dataset"))

['images', 'customers.csv', 'sample_submission.csv', 'transactions_train.csv', 'articles.csv']


#Loading the Dataset into PySpark

#Define the Dataset Path

In [ ]:
DATA_PATH = "/content/hm_dataset"

#Load customers.csv

In [ ]:
customers_df = spark.read.csv(
    f"{DATA_PATH}/customers.csv",
    header=True,
    inferSchema=True
)

#Load articles.csv

In [ ]:
articles_df = spark.read.csv(
    f"{DATA_PATH}/articles.csv",
    header=True,
    inferSchema=True
)

#Load transactions_train.csv

In [ ]:
transactions_df = spark.read.csv(
    f"{DATA_PATH}/transactions_train.csv",
    header=True,
    inferSchema=True
)

#Inspect the Data

In [ ]:
customers_df.show(5)

+--------------------+----+------+------------------+----------------------+---+--------------------+
|         customer_id|  FN|Active|club_member_status|fashion_news_frequency|age|         postal_code|
+--------------------+----+------+------------------+----------------------+---+--------------------+
|00000dbacae5abe5e...|NULL|  NULL|            ACTIVE|                  NONE| 49|52043ee2162cf5aa7...|
|0000423b00ade9141...|NULL|  NULL|            ACTIVE|                  NONE| 25|2973abc54daa8a5f8...|
|000058a12d5b43e67...|NULL|  NULL|            ACTIVE|                  NONE| 24|64f17e6a330a85798...|
|00005ca1c9ed5f514...|NULL|  NULL|            ACTIVE|                  NONE| 54|5d36574f52495e81f...|
|00006413d8573cd20...| 1.0|   1.0|            ACTIVE|             Regularly| 52|25fa5ddee9aac01b3...|
+--------------------+----+------+------------------+----------------------+---+--------------------+
only showing top 5 rows


In [ ]:
customers_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- FN: double (nullable = true)
 |-- Active: double (nullable = true)
 |-- club_member_status: string (nullable = true)
 |-- fashion_news_frequency: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- postal_code: string (nullable = true)



In [ ]:
customers_df.count()

1371980

In [ ]:
articles_df.show(5)

+----------+------------+-----------------+---------------+-----------------+------------------+-----------------------+-------------------------+-----------------+-----------------+-------------------------+---------------------------+--------------------------+----------------------------+-------------+---------------+----------+----------------+--------------+----------------+----------+--------------------+----------------+------------------+--------------------+
|article_id|product_code|        prod_name|product_type_no|product_type_name|product_group_name|graphical_appearance_no|graphical_appearance_name|colour_group_code|colour_group_name|perceived_colour_value_id|perceived_colour_value_name|perceived_colour_master_id|perceived_colour_master_name|department_no|department_name|index_code|      index_name|index_group_no|index_group_name|section_no|        section_name|garment_group_no|garment_group_name|         detail_desc|
+----------+------------+-----------------+-------------

In [ ]:
articles_df.printSchema()

root
 |-- article_id: integer (nullable = true)
 |-- product_code: integer (nullable = true)
 |-- prod_name: string (nullable = true)
 |-- product_type_no: integer (nullable = true)
 |-- product_type_name: string (nullable = true)
 |-- product_group_name: string (nullable = true)
 |-- graphical_appearance_no: integer (nullable = true)
 |-- graphical_appearance_name: string (nullable = true)
 |-- colour_group_code: integer (nullable = true)
 |-- colour_group_name: string (nullable = true)
 |-- perceived_colour_value_id: integer (nullable = true)
 |-- perceived_colour_value_name: string (nullable = true)
 |-- perceived_colour_master_id: integer (nullable = true)
 |-- perceived_colour_master_name: string (nullable = true)
 |-- department_no: integer (nullable = true)
 |-- department_name: string (nullable = true)
 |-- index_code: string (nullable = true)
 |-- index_name: string (nullable = true)
 |-- index_group_no: integer (nullable = true)
 |-- index_group_name: string (nullable = true)

In [ ]:
articles_df.count()

105542

In [ ]:
transactions_df.show(5)

+----------+--------------------+----------+--------------------+----------------+
|     t_dat|         customer_id|article_id|               price|sales_channel_id|
+----------+--------------------+----------+--------------------+----------------+
|2018-09-20|000058a12d5b43e67...| 663713001|0.050830508474576264|               2|
|2018-09-20|000058a12d5b43e67...| 541518023| 0.03049152542372881|               2|
|2018-09-20|00007d2de826758b6...| 505221004| 0.01523728813559322|               2|
|2018-09-20|00007d2de826758b6...| 685687003|0.016932203389830508|               2|
|2018-09-20|00007d2de826758b6...| 685687004|0.016932203389830508|               2|
+----------+--------------------+----------+--------------------+----------------+
only showing top 5 rows


In [ ]:
transactions_df.printSchema()

root
 |-- t_dat: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- article_id: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- sales_channel_id: integer (nullable = true)



In [ ]:
transactions_df.count()

31788324

#Missing Values in Customers

In [ ]:
from pyspark.sql.functions import col, count, when

customers_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in customers_df.columns
]).show()

+-----------+------+------+------------------+----------------------+-----+-----------+
|customer_id|    FN|Active|club_member_status|fashion_news_frequency|  age|postal_code|
+-----------+------+------+------------------+----------------------+-----+-----------+
|          0|895050|907576|              6062|                 16009|15861|          0|
+-----------+------+------+------------------+----------------------+-----+-----------+



#Missing Values in Articles

In [ ]:
articles_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in articles_df.columns
]).show()

+----------+------------+---------+---------------+-----------------+------------------+-----------------------+-------------------------+-----------------+-----------------+-------------------------+---------------------------+--------------------------+----------------------------+-------------+---------------+----------+----------+--------------+----------------+----------+------------+----------------+------------------+-----------+
|article_id|product_code|prod_name|product_type_no|product_type_name|product_group_name|graphical_appearance_no|graphical_appearance_name|colour_group_code|colour_group_name|perceived_colour_value_id|perceived_colour_value_name|perceived_colour_master_id|perceived_colour_master_name|department_no|department_name|index_code|index_name|index_group_no|index_group_name|section_no|section_name|garment_group_no|garment_group_name|detail_desc|
+----------+------------+---------+---------------+-----------------+------------------+-----------------------+------

#Missing Values in Transactions

In [ ]:
transactions_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in transactions_df.columns
]).show()

+-----+-----------+----------+-----+----------------+
|t_dat|customer_id|article_id|price|sales_channel_id|
+-----+-----------+----------+-----+----------------+
|    0|          0|         0|    0|               0|
+-----+-----------+----------+-----+----------------+



#inspect the unique values in customers

In [ ]:
customers_df.groupBy("club_member_status").count().orderBy("count", ascending=False).show()

+------------------+-------+
|club_member_status|  count|
+------------------+-------+
|            ACTIVE|1272491|
|        PRE-CREATE|  92960|
|              NULL|   6062|
|         LEFT CLUB|    467|
+------------------+-------+



In [ ]:
customers_df.groupBy("fashion_news_frequency").count().orderBy("count", ascending=False).show()

+----------------------+------+
|fashion_news_frequency| count|
+----------------------+------+
|                  NONE|877711|
|             Regularly|477416|
|                  NULL| 16009|
|               Monthly|   842|
|                  None|     2|
+----------------------+------+



In [ ]:
customers_df.groupBy("FN").count().show()

+----+------+
|  FN| count|
+----+------+
|NULL|895050|
| 1.0|476930|
+----+------+



In [ ]:
customers_df.groupBy("Active").count().show()

+------+------+
|Active| count|
+------+------+
|  NULL|907576|
|   1.0|464404|
+------+------+



#Clean the Customers Dataset

In [ ]:
from pyspark.sql.functions import col, when

#Standardize fashion_news_frequency (we found NONE and None)

In [ ]:
customers_df = customers_df.withColumn(
    "fashion_news_frequency",
    when(col("fashion_news_frequency") == "None", "NONE")
    .otherwise(col("fashion_news_frequency"))
)

#Fill Missing Values

In [ ]:
customers_df = customers_df.fillna({
    "FN": 0,
    "Active": 0,
    "club_member_status": "UNKNOWN",
    "fashion_news_frequency": "NONE"
})

#Compute the Median Age

In [ ]:
median_age = customers_df.approxQuantile("age", [0.5], 0.01)[0]

print("Median Age:", median_age)

Median Age: 32.0


#Fill Missing Ages

In [ ]:
customers_df = customers_df.fillna({
    "age": median_age
})

#Verify the Cleaning

In [ ]:
from pyspark.sql.functions import count, when

customers_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in customers_df.columns
]).show()

+-----------+---+------+------------------+----------------------+---+-----------+
|customer_id| FN|Active|club_member_status|fashion_news_frequency|age|postal_code|
+-----------+---+------+------------------+----------------------+---+-----------+
|          0|  0|     0|                 0|                     0|  0|          0|
+-----------+---+------+------------------+----------------------+---+-----------+



In [ ]:
from pyspark.sql.functions import col

customers_df = (
    customers_df
    .withColumn("FN", col("FN").cast("int"))
    .withColumn("Active", col("Active").cast("int"))
)

In [ ]:
customers_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- FN: integer (nullable = true)
 |-- Active: integer (nullable = true)
 |-- club_member_status: string (nullable = false)
 |-- fashion_news_frequency: string (nullable = false)
 |-- age: integer (nullable = true)
 |-- postal_code: string (nullable = true)



#Clean the Articles Dataset

In [ ]:
articles_df = articles_df.fillna({
    "detail_desc": "No Description Available"
})

In [ ]:
from pyspark.sql.functions import count, when, col

articles_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in articles_df.columns
]).show()

+----------+------------+---------+---------------+-----------------+------------------+-----------------------+-------------------------+-----------------+-----------------+-------------------------+---------------------------+--------------------------+----------------------------+-------------+---------------+----------+----------+--------------+----------------+----------+------------+----------------+------------------+-----------+
|article_id|product_code|prod_name|product_type_no|product_type_name|product_group_name|graphical_appearance_no|graphical_appearance_name|colour_group_code|colour_group_name|perceived_colour_value_id|perceived_colour_value_name|perceived_colour_master_id|perceived_colour_master_name|department_no|department_name|index_code|index_name|index_group_no|index_group_name|section_no|section_name|garment_group_no|garment_group_name|detail_desc|
+----------+------------+---------+---------------+-----------------+------------------+-----------------------+------

#Check Duplicate Transactions

In [ ]:
total_rows = transactions_df.count()
distinct_rows = transactions_df.distinct().count()

print("Total Rows :", total_rows)
print("Distinct Rows :", distinct_rows)
print("Duplicate Rows :", total_rows - distinct_rows)

Total Rows : 31788324
Distinct Rows : 28813419
Duplicate Rows : 2974905


#Find the Date Range

In [ ]:
from pyspark.sql.functions import min, max

transactions_df.select(
    min("t_dat").alias("Start_Date"),
    max("t_dat").alias("End_Date")
).show()

+----------+----------+
|Start_Date|  End_Date|
+----------+----------+
|2018-09-20|2020-09-22|
+----------+----------+



#Count Unique Customers

In [ ]:
from pyspark.sql.functions import countDistinct

transactions_df.select(
    countDistinct("customer_id").alias("Unique_Customers")
).show()

+----------------+
|Unique_Customers|
+----------------+
|         1362281|
+----------------+



#Count Unique Products

In [ ]:
transactions_df.select(
    countDistinct("article_id").alias("Unique_Articles")
).show()

+---------------+
|Unique_Articles|
+---------------+
|         104547|
+---------------+



#Metric	Value
Total Transactions	31,788,324,
Distinct Transaction Rows	28,813,419,
Duplicate Records	2,974,905,
Purchase History	2 years,
Customers with Purchases	1,362,281,
Products Purchased	104,547

#Save the Cleaned Data

In [ ]:
PROCESSED_PATH = "/content/drive/MyDrive/Recommendation_Engine/data/processed"

In [ ]:
customers_df.write.mode("overwrite").parquet(
    f"{PROCESSED_PATH}/customers_clean.parquet"
)

In [ ]:
articles_df.write.mode("overwrite").parquet(
    f"{PROCESSED_PATH}/articles_clean.parquet"
)

In [ ]:
transactions_df.write.mode("overwrite").parquet(
    f"{PROCESSED_PATH}/transactions_clean.parquet"
)

#Identify Cold-Start Users

In [ ]:
from pyspark.sql.functions import col

# Customers who made at least one purchase
purchased_customers = transactions_df.select("customer_id").distinct()

# Customers with no purchase history
cold_start_users = customers_df.join(
    purchased_customers,
    on="customer_id",
    how="left_anti"
)

print("Cold Start Users:", cold_start_users.count())

Cold Start Users: 9699


In [ ]:
cold_start_users.show(5)

+--------------------+---+------+------------------+----------------------+---+--------------------+
|         customer_id| FN|Active|club_member_status|fashion_news_frequency|age|         postal_code|
+--------------------+---+------+------------------+----------------------+---+--------------------+
|0033ed9017159dac2...|  1|     1|            ACTIVE|             Regularly| 20|2c29ae653a9282cce...|
|00f53046607c22cd6...|  0|     0|            ACTIVE|                  NONE| 17|d8983094daae78cec...|
|01587bfe37f402820...|  0|     0|            ACTIVE|                  NONE| 37|2c29ae653a9282cce...|
|0166011fdf2b9d363...|  1|     1|            ACTIVE|             Regularly| 52|73718e25b2ad62deb...|
|024d68550b2390c3c...|  1|     1|            ACTIVE|             Regularly| 32|2c29ae653a9282cce...|
+--------------------+---+------+------------------+----------------------+---+--------------------+
only showing top 5 rows


#Identify Cold-Start Items

In [ ]:
purchased_articles = transactions_df.select("article_id").distinct()

cold_start_items = articles_df.join(
    purchased_articles,
    on="article_id",
    how="left_anti"
)

print("Cold Start Items:", cold_start_items.count())

Cold Start Items: 995


In [ ]:
cold_start_items.select(
    "article_id",
    "prod_name",
    "product_type_name"
).show(5, truncate=False)

+----------+----------------------+------------------------+
|article_id|prod_name             |product_type_name       |
+----------+----------------------+------------------------+
|187949032 |Padded pyjama         |Pyjama jumpsuit/playsuit|
|288859020 |Kakan 2-p cableknit BG|Underwear Tights        |
|395730045 |VIOLA 2-pack (TVP)    |Polo shirt              |
|462435036 |6P Tanktop Body       |Bodysuit                |
|485689035 |Bobby l/l pj BB       |Pyjama set              |
+----------+----------------------+------------------------+
only showing top 5 rows


In [ ]:
cold_start_users.write \
    .mode("overwrite") \
    .parquet(f"{PROCESSED_PATH}/cold_start_users.parquet")

In [ ]:
cold_start_items.write \
    .mode("overwrite") \
    .parquet(f"{PROCESSED_PATH}/cold_start_items.parquet")

In [ ]:
import os

print(os.listdir(PROCESSED_PATH))

['customers_clean.parquet', 'articles_clean.parquet', 'transactions_clean.parquet', 'cold_start_users.parquet', 'cold_start_items.parquet']


#Load the Cleaned Data

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("HM-Recommendation-System") \
    .getOrCreate()

PROCESSED_PATH = "/content/drive/MyDrive/Recommendation_Engine/data/processed"

customers_df = spark.read.parquet(f"{PROCESSED_PATH}/customers_clean.parquet")
articles_df = spark.read.parquet(f"{PROCESSED_PATH}/articles_clean.parquet")
transactions_df = spark.read.parquet(f"{PROCESSED_PATH}/transactions_clean.parquet")

In [3]:
customers_df.count(), articles_df.count(), transactions_df.count()

(1371980, 105542, 31788324)

#Engineer Contextual Features

#Time Features

In [4]:
from pyspark.sql.functions import (
    year,
    month,
    dayofmonth,
    quarter,
    dayofweek
)

In [5]:
transactions_df = (
    transactions_df
    .withColumn("purchase_year", year("t_dat"))
    .withColumn("purchase_month", month("t_dat"))
    .withColumn("purchase_day", dayofmonth("t_dat"))
    .withColumn("purchase_quarter", quarter("t_dat"))
    .withColumn("purchase_dayofweek", dayofweek("t_dat"))
)

In [6]:
transactions_df.select(
    "t_dat",
    "purchase_year",
    "purchase_month",
    "purchase_day",
    "purchase_quarter",
    "purchase_dayofweek"
).show(10, truncate=False)

+----------+-------------+--------------+------------+----------------+------------------+
|t_dat     |purchase_year|purchase_month|purchase_day|purchase_quarter|purchase_dayofweek|
+----------+-------------+--------------+------------+----------------+------------------+
|2019-11-29|2019         |11            |29          |4               |6                 |
|2019-11-29|2019         |11            |29          |4               |6                 |
|2019-11-29|2019         |11            |29          |4               |6                 |
|2019-11-29|2019         |11            |29          |4               |6                 |
|2019-11-29|2019         |11            |29          |4               |6                 |
|2019-11-29|2019         |11            |29          |4               |6                 |
|2019-11-29|2019         |11            |29          |4               |6                 |
|2019-11-29|2019         |11            |29          |4               |6                 |

In [7]:
from pyspark.sql.functions import max

reference_date = transactions_df.select(
    max("t_dat").alias("reference_date")
).collect()[0]["reference_date"]

print("Reference Date:", reference_date)

Reference Date: 2020-09-22


In [8]:
from pyspark.sql.functions import max

last_purchase_df = (
    transactions_df
    .groupBy("customer_id")
    .agg(
        max("t_dat").alias("last_purchase_date")
    )
)

last_purchase_df.show(10, truncate=False)

+----------------------------------------------------------------+------------------+
|customer_id                                                     |last_purchase_date|
+----------------------------------------------------------------+------------------+
|ab1e1d2cfc55578021587acad6e035fa04a53838d417b83db78a5ce2d12be849|2020-09-19        |
|ac4880bdd0397bdc9cac31b29c7906e1d543ae60653807dce02ec7e249cdfa46|2020-02-27        |
|acd1eb2c1cd341126563eff0cf751b884f0b46496a5b791dd4b918c62437aa3f|2020-09-19        |
|af354824485c5f88562b1dcacaa13febdfc61648b3386bf011cc48d0a6dbb400|2020-07-01        |
|afa988d12ee3919f4fc32f0cf76488ac1d8315c1fd46258edb11abcd21a6020d|2020-09-05        |
|b2201048684715949316e27ebe8f51c13c47a7535691d6ed833d51a8cc299552|2020-06-20        |
|b2682c5d7d2f74466149c7eb398a9b3c5161c17e7d113d7ada614a457b728e3e|2020-04-11        |
|b3afb17c757a4e62964ca348b69a285be952458a374b18018bf9810cc4f583c0|2020-09-15        |
|b440a9b9fae9b00e829fb35ab228b138019329bf9e7b171ac2acb

#Calculate Recency

In [9]:
from pyspark.sql.functions import datediff, lit

In [10]:
recency_df = (
    last_purchase_df
    .withColumn(
        "recency_days",
        datediff(
            lit(reference_date),
            "last_purchase_date"
        )
    )
)

recency_df.show(10, truncate=False)

+----------------------------------------------------------------+------------------+------------+
|customer_id                                                     |last_purchase_date|recency_days|
+----------------------------------------------------------------+------------------+------------+
|ab1e1d2cfc55578021587acad6e035fa04a53838d417b83db78a5ce2d12be849|2020-09-19        |3           |
|ac4880bdd0397bdc9cac31b29c7906e1d543ae60653807dce02ec7e249cdfa46|2020-02-27        |208         |
|acd1eb2c1cd341126563eff0cf751b884f0b46496a5b791dd4b918c62437aa3f|2020-09-19        |3           |
|af354824485c5f88562b1dcacaa13febdfc61648b3386bf011cc48d0a6dbb400|2020-07-01        |83          |
|afa988d12ee3919f4fc32f0cf76488ac1d8315c1fd46258edb11abcd21a6020d|2020-09-05        |17          |
|b2201048684715949316e27ebe8f51c13c47a7535691d6ed833d51a8cc299552|2020-06-20        |94          |
|b2682c5d7d2f74466149c7eb398a9b3c5161c17e7d113d7ada614a457b728e3e|2020-04-11        |164         |
|b3afb17c7

#Product Popularity

In [11]:
from pyspark.sql.functions import count

product_popularity_df = (
    transactions_df
    .groupBy("article_id")
    .agg(
        count("*").alias("purchase_count")
    )
)

In [12]:
product_popularity_df.orderBy(
    "purchase_count",
    ascending=False
).show(10)

+----------+--------------+
|article_id|purchase_count|
+----------+--------------+
| 706016001|         50287|
| 706016002|         35043|
| 372860001|         31718|
| 610776002|         30199|
| 759871002|         26329|
| 464297007|         25025|
| 372860002|         24458|
| 610776001|         22451|
| 399223001|         22236|
| 706016003|         21241|
+----------+--------------+
only showing top 10 rows


#Calculate Monthly Product Popularity

In [13]:
from pyspark.sql.functions import count

monthly_product_popularity_df = (
    transactions_df
    .groupBy(
        "article_id",
        "purchase_year",
        "purchase_month"
    )
    .agg(
        count("*").alias("monthly_purchase_count")
    )
)

In [14]:
monthly_product_popularity_df.orderBy(
    "article_id",
    "purchase_year",
    "purchase_month"
).show(20, truncate=False)

+----------+-------------+--------------+----------------------+
|article_id|purchase_year|purchase_month|monthly_purchase_count|
+----------+-------------+--------------+----------------------+
|108775015 |2018         |9             |662                   |
|108775015 |2018         |10            |1532                  |
|108775015 |2018         |11            |1660                  |
|108775015 |2018         |12            |1207                  |
|108775015 |2019         |1             |1522                  |
|108775015 |2019         |2             |1198                  |
|108775015 |2019         |3             |1131                  |
|108775015 |2019         |4             |1137                  |
|108775015 |2019         |5             |513                   |
|108775015 |2019         |6             |97                    |
|108775015 |2019         |7             |47                    |
|108775015 |2019         |8             |27                    |
|108775015 |2019         

#Save the Feature Tables

In [15]:
FEATURE_PATH = "/content/drive/MyDrive/Recommendation_Engine/features"

In [16]:
recency_df.write \
    .mode("overwrite") \
    .parquet(f"{FEATURE_PATH}/recency.parquet")

In [17]:
product_popularity_df.write \
    .mode("overwrite") \
    .parquet(f"{FEATURE_PATH}/product_popularity.parquet")

In [18]:
monthly_product_popularity_df.write \
    .mode("overwrite") \
    .parquet(f"{FEATURE_PATH}/monthly_product_popularity.parquet")

In [19]:
import os

print(os.listdir(FEATURE_PATH))

['recency.parquet', 'product_popularity.parquet', 'monthly_product_popularity.parquet']


#Customer Vocabulary

In [20]:
customer_vocab = (
    customers_df
    .select("customer_id")
    .distinct()
    .orderBy("customer_id")
)

print("Number of unique customers:", customer_vocab.count())

customer_vocab.show(10, truncate=False)

Number of unique customers: 1371980
+----------------------------------------------------------------+
|customer_id                                                     |
+----------------------------------------------------------------+
|00000dbacae5abe5e23885899a1fa44253a17956c6d1c3d25f88aa139fdfc657|
|0000423b00ade91418cceaf3b26c6af3dd342b51fd051eec9c12fb36984420fa|
|000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318|
|00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2c5feb1ca5dff07c43e|
|00006413d8573cd20ed7128e53b7b13819fe5cfc2d801fe7fc0f26dd8d65a85a|
|000064249685c11552da43ef22a5030f35a147f723d5b02ddd9fd22452b1f5a6|
|0000757967448a6cb83efb3ea7a3fb9d418ac7adf2379d8cd0c725276a467a2a|
|00007d2de826758b65a93dd24ce629ed66842531df6699338c5570910a014cc2|
|00007e8d4e54114b5b2a9b51586325a8d0fa74ea23ef77334eaec4ffccd7ebcc|
|00008469a21b50b3d147c97135e25b4201a8c58997f78782a0cc706645e14493|
+----------------------------------------------------------------+
only showing top 10 rows


#Article Vocabulary

In [21]:
article_vocab = (
    articles_df
    .select("article_id")
    .distinct()
    .orderBy("article_id")
)

print("Number of unique articles:", article_vocab.count())

article_vocab.show(10)

Number of unique articles: 105542
+----------+
|article_id|
+----------+
| 108775015|
| 108775044|
| 108775051|
| 110065001|
| 110065002|
| 110065011|
| 111565001|
| 111565003|
| 111586001|
| 111593001|
+----------+
only showing top 10 rows


#Product Type Vocabulary

In [22]:
product_type_vocab = (
    articles_df
    .select("product_type_name")
    .distinct()
    .orderBy("product_type_name")
)

print("Unique Product Types:", product_type_vocab.count())

product_type_vocab.show(20, truncate=False)

Unique Product Types: 131
+-----------------+
|product_type_name|
+-----------------+
|Accessories set  |
|Alice band       |
|Baby Bib         |
|Backpack         |
|Bag              |
|Ballerinas       |
|Beanie           |
|Belt             |
|Bikini top       |
|Blanket          |
|Blazer           |
|Blouse           |
|Bodysuit         |
|Bootie           |
|Boots            |
|Bra              |
|Bra extender     |
|Bracelet         |
|Braces           |
|Bucket hat       |
+-----------------+
only showing top 20 rows


#Department Vocabulary

In [23]:
department_vocab = (
    articles_df
    .select("department_name")
    .distinct()
    .orderBy("department_name")
)

print("Unique Departments:", department_vocab.count())

department_vocab.show(20, truncate=False)

Unique Departments: 250
+-------------------------+
|department_name          |
+-------------------------+
|AK Bottoms               |
|AK Dresses & Outdoor     |
|AK Other                 |
|AK Tops Jersey & Woven   |
|AK Tops Knitwear         |
|Accessories              |
|Accessories Boys         |
|Accessories Other        |
|Asia Assortment          |
|Baby Boy Jersey Fancy    |
|Baby Boy Knitwear        |
|Baby Boy Local Relevance |
|Baby Boy Outdoor         |
|Baby Boy Woven           |
|Baby Exclusive           |
|Baby Girl Jersey Fancy   |
|Baby Girl Knitwear       |
|Baby Girl Local Relevance|
|Baby Girl Outdoor        |
|Baby Girl Woven          |
+-------------------------+
only showing top 20 rows


#Color Vocabulary

In [24]:
color_vocab = (
    articles_df
    .select("colour_group_name")
    .distinct()
    .orderBy("colour_group_name")
)

print("Unique Colors:", color_vocab.count())

color_vocab.show(20, truncate=False)

Unique Colors: 50
+-----------------+
|colour_group_name|
+-----------------+
|Beige            |
|Black            |
|Blue             |
|Bronze/Copper    |
|Dark Beige       |
|Dark Blue        |
|Dark Green       |
|Dark Grey        |
|Dark Orange      |
|Dark Pink        |
|Dark Purple      |
|Dark Red         |
|Dark Turquoise   |
|Dark Yellow      |
|Gold             |
|Green            |
|Greenish Khaki   |
|Grey             |
|Greyish Beige    |
|Light Beige      |
+-----------------+
only showing top 20 rows


In [25]:
VOCAB_PATH = "/content/drive/MyDrive/Recommendation_Engine/vocabularies"

In [26]:
customer_vocab.write.mode("overwrite").parquet(
    f"{VOCAB_PATH}/customer_vocab.parquet"
)

In [27]:
article_vocab.write.mode("overwrite").parquet(
    f"{VOCAB_PATH}/article_vocab.parquet"
)

In [28]:
product_type_vocab.write.mode("overwrite").parquet(
    f"{VOCAB_PATH}/product_type_vocab.parquet"
)

In [29]:
department_vocab.write.mode("overwrite").parquet(
    f"{VOCAB_PATH}/department_vocab.parquet"
)

In [30]:
color_vocab.write.mode("overwrite").parquet(
    f"{VOCAB_PATH}/color_vocab.parquet"
)

In [31]:
import os

print(os.listdir(VOCAB_PATH))

['customer_vocab.parquet', 'article_vocab.parquet', 'product_type_vocab.parquet', 'department_vocab.parquet', 'color_vocab.parquet']
